# Observations

The Peruvian government maintains a database of disaster impact observations. In this notebook we use an extract from the database to identify past events that affected Ica.

The extract in the observations_raw.csv file has already been processed (to reduce the size, selecting Ica events) and reformatted (to add English column names).

This notebook:
- Filters observations to the modelled region
- Aggregates reports to event totals
- Adjusts for population growth and inflation to get present-day equivalent impacts
- Estimates a return period for each event
- Writes these to file

In [19]:
from pathlib import Path

# -----------------------------------------------
# Set up paths to data
# -----------------------------------------------

OBSERVATIONS_DIR = "../data/observations/"
OBSERVATIONS_FILE = "observations_raw.csv"
observations_filepath = Path(OBSERVATIONS_DIR) / OBSERVATIONS_FILE


In [20]:
import pandas as pd
import numpy as np
from IPython.display import display

# -----------------------------------------------
# Read and reformat the raw observations
# -----------------------------------------------

obs = pd.read_csv(observations_filepath)
obs = obs.rename(columns={
    "EMERGENCY CODE (CÓDIGO DE EMERGENCIA-SINPAD)": "code",
    "DATE OF THE EMERGENCY (FECHA  DE LA EMER)": "date",
    "Year (AÑO)": "year",
    "Month (MES)": "month",
    "COD. DISTRITO": "district_code",
    "Department - Admin 1 (DPTO.)": "department",
    "Province - Admin 2 (PROV.)": "province",
    "District - Admin 3 (DIST.)": "district",
    "Deaths (FALLECIDOS)": "deaths",
    "Missing (DESAPARECIDOS)": "missing",
    "Injured (HERIDOS)": "injured",
    "VICTIMS (DAMNIFICADOS)": "victims",
    "AFFECTED (AFECTADOS)": "affected",
    "HOMES DESTROYED (VIVIENDAS DESTRUIDAS)": "homes_destroyed",
    "AFFECTED HOMES (VIVIENDAS AFECTADAS)": "homes_affected",
})
# Subset to the columns we need:
obs = obs[[
    "date", "year", "month", "district", "homes_destroyed", "homes_affected", "victims", "affected"
]]
obs["affected_or_destroyed"] = obs["homes_destroyed"] + obs["homes_affected"]
obs["victims_or_affected"] = obs["victims"] + obs["affected"]


# -----------------------------------------------
# Filter to modelled districts
# -----------------------------------------------

# A manual check of the overlap of Ica districts and the study area covered by the hazard show these Ica districts to be
# outside of the hazard area. Therefore we can't model their losses and they should be dropped from the observations.
districts_to_drop = [
    "SAN JOSÉ DE LOS MOLINOS",
    "GUADALUPE",
    "PACHACUTEC",
    "SANTIAGO",
    "OCUCAJE"
]
obs = obs[~obs["district"].isin(districts_to_drop)]
print("Relevant observations:")
display(obs)


Relevant observations:


,date,year,month,district,homes_destroyed,homes_affected,victims,affected,affected_or_destroyed,victims_or_affected
0,12/12/2003,2003,Diciembre,ICA,1,3,11,13,4,24
1,21/01/2004,2004,Enero,ICA,0,9,0,45,9,45
2,07/11/2004,2004,Noviembre,PARCONA,0,2,0,13,2,13
3,05/11/2005,2005,Noviembre,ICA,0,2,0,6,2,6
4,11/01/2005,2005,Enero,SALAS,0,3,0,10,3,10
5,18/03/2006,2006,Marzo,ICA,0,3,0,0,3,0
6,22/12/2007,2007,Diciembre,ICA,0,5,0,0,5,0
7,02/04/2007,2007,Abril,LOS AQUIJES,0,6,0,17,6,17
9,05/06/2008,2008,Junio,ICA,5,1,22,1,6,23
10,08/02/2011,2011,Febrero,LOS AQUIJES,0,69,0,348,69,348


In [21]:
# -------------------------------------------------
# Group events in time and aggregate over districts
# -------------------------------------------------

# Use the date column to identify events that happen within 60 days of each other: we classify these as the same event.
# This is a subjective choice!
grouping_window_days = 60
obs["date"] = pd.to_datetime(obs["date"], format="%d/%m/%Y")
obs = obs.sort_values("date").reset_index(drop=True)
obs["event_id"] = (obs["date"].diff().dt.days > grouping_window_days).cumsum()

# Group the events and sum them over districts to get total impacts
obs = obs.drop(columns=["district", "date"]).groupby(["event_id"]).agg({
    "homes_destroyed": "sum",
    "homes_affected": "sum",
    "affected_or_destroyed": "sum",
    "victims": "sum",
    "affected": "sum",
    "victims_or_affected": "sum",
    "year": "first",
    "month": "first"
}).reset_index()

print("Aggregated observations:")
display(obs)


Aggregated observations:


,event_id,homes_destroyed,homes_affected,affected_or_destroyed,victims,affected,victims_or_affected,year,month
0,0,1,12,13,11,58,69,2003,Diciembre
1,1,0,2,2,0,13,13,2004,Noviembre
2,2,0,3,3,0,10,10,2005,Enero
3,3,0,2,2,0,6,6,2005,Noviembre
4,4,0,3,3,0,0,0,2006,Marzo
5,5,0,6,6,0,17,17,2007,Abril
6,6,0,5,5,0,0,0,2007,Diciembre
7,7,5,1,6,22,1,23,2008,Junio
8,8,0,69,69,0,348,348,2011,Febrero
9,9,0,0,0,0,0,0,2012,Febrero


In [22]:
# -----------------------------------------------
# Drop small events
# -----------------------------------------------

# We choose to filter to events with at least 10 affected or destroyed homes to focus on more significant events
# This avoids issues with underreporting of very small events, and handling small events that may only be included in 
# the database because of much larger impacts in neighbouring regions.
obs = obs[obs["affected_or_destroyed"] >= 10].reset_index(drop=True)
print("Filtered observations:")
display(obs)

Filtered observations:


,event_id,homes_destroyed,homes_affected,affected_or_destroyed,victims,affected,victims_or_affected,year,month
0,0,1,12,13,11,58,69,2003,Diciembre
1,8,0,69,69,0,348,348,2011,Febrero
2,12,134,4385,4519,713,21925,22638,2017,Enero


### Adjust for historic growth

The observation data we have reflects impacts in past years. Since then the population of Ica and its socioeconomic conditions have changed. We would like to adjust the historic observations to reflect what the impacts would look like if they happened today.

In this calculation the adjustment is very simple: we scale the number of houses by the national change in population since the event. A more advanced adjustment would use Ica-specific population data and might also take into account changes in average household size or even spatial changes in housing.

In [23]:
SOCIOECONOMIC_DIR = "../data/socioeconomic_change/"
POP_CHANGE_FILE = "population_change_historic.csv"
PRESENT_YEAR = 2025

df_pop_change = pd.read_csv(Path(SOCIOECONOMIC_DIR) / POP_CHANGE_FILE)
df_pop_change['scale'] = df_pop_change['pop_change'] + 1

# Define a method that calculates the adustment we need to apply between any two years
def population_ratio(year_from, year_to):
    assert year_from <= year_to, "year_from must be less than or equal to year_to"
    assert year_from in df_pop_change["year"].values, f"year_from {year_from} not found in population change data"
    assert year_to in df_pop_change["year"].values, f"year_to {year_to} not found in population change data"
    df_filtered = df_pop_change[(df_pop_change["year"] >= year_from) & (df_pop_change["year"] <= year_to)]
    ratio = df_filtered["scale"].prod()
    return ratio

obs["population_ratio"] = obs.apply(lambda row: population_ratio(row["year"], PRESENT_YEAR), axis=1)

for col in ["homes_affected", "homes_destroyed", "affected_or_destroyed", "victims", "affected", "victims_or_affected"]:
    obs[f"{col}_present"] = obs[col] * obs["population_ratio"]

obs.drop(columns=["population_ratio", "homes_affected", "homes_destroyed", "affected_or_destroyed", "victims", "affected", "victims_or_affected"], inplace=True)
display(display(obs.style.format({
    "homes_affected_present": "{:,.1f}",
    "homes_destroyed_present": "{:,.1f}",
    "affected_or_destroyed_present": "{:,.1f}",
    "victims_present": "{:,.1f}",
    "affected_present": "{:,.1f}",
    "victims_or_affected_present": "{:,.1f}"
})))

,event_id,year,month,homes_affected_present,homes_destroyed_present,affected_or_destroyed_present,victims_present,affected_present,victims_or_affected_present
0,0,2003,Diciembre,15.2,1.3,16.4,13.9,73.4,87.3
1,8,2011,Febrero,82.1,0.0,82.1,0.0,413.9,413.9
2,12,2017,Enero,"4,897.4",149.7,"5,047.1",796.3,"24,487.1","25,283.5"


None

### Estimate historic losses

From the 2017 disaster recovery reports, we have information on the post-disaster government expenditure for housing reconstruction. The costs include costs for repairing damaged houses, rebuilding destroyed houses, and purchasing land a building new houses for houses that were deemed to be at high risk of future flooding. We can use this information to rank their severity

Note: this is not the full economic cost of losses, this is what the government spent.

In [24]:
RECOVERY_COSTS_USD = {
    'affected': 4260,   # Cost of rebuilding a damaged home, including improvements to resilience
    'destroyed': 6200,  # Cost of replacing a home, with improved standard of building materials
    'resettled': 17000  # Cost of buying new land and building a new home for buildings in high risk areas. 
                        # (We don't have information on the criteria for this or the number of homes that qualified in past events).   
}

obs["recovery_cost_usd"] = obs["homes_affected_present"] * RECOVERY_COSTS_USD['affected'] + obs["homes_destroyed_present"] * RECOVERY_COSTS_USD['destroyed']
display(obs.style.format({
    "homes_affected_present": "{:,.1f}",
    "homes_destroyed_present": "{:,.1f}",
    "affected_or_destroyed_present": "{:,.1f}",
    "recovery_cost_usd": "${:,.0f}",
    "victims_present": "{:,.1f}",
    "affected_present": "{:,.1f}",
    "victims_or_affected_present": "{:,.1f}"
}))

,event_id,year,month,homes_affected_present,homes_destroyed_present,affected_or_destroyed_present,victims_present,affected_present,victims_or_affected_present,recovery_cost_usd
0,0,2003,Diciembre,15.2,1.3,16.4,13.9,73.4,87.3,"$72,494"
1,8,2011,Febrero,82.1,0.0,82.1,0.0,413.9,413.9,"$349,611"
2,12,2017,Enero,"4,897.4",149.7,"5,047.1",796.3,"24,487.1","25,283.5","$21,790,936"


### Estimate observation return periods

Since our flood hazard data is provided by return period, our observations need return periods in order for us to be able to compare modelled losses.

Our observational dataset covers 18 years of historic observations (2003-2020). Without more detailed historic information, we assume the 18 years of historic data represent the present day climate, and that we can assign return periods based on this (ranked by estimated recovery cost).

However, for the most severe event, the 2017 floods, we have media reports and analyses from the time that give return period estimates for its intensity, and we will use these estimates instead. There have been many different estimates of return periods, or stating that this was the "most severe event since X", and most of them refer to the national impacts rather than just Ica. Taking these into account we are estimating a return period of 92 years for the event, as this was the most severe El Niño since 1925.

However it is also very possible that this is higher or lower – adjust the parameter in the following code and see how this affects the calibration.

In [25]:
N_YEARS_OBSERVATIONS = 18
RETURN_PERIOD_2017 = 92

# Sort by event severity
obs = obs.sort_values("recovery_cost_usd", ascending=False).reset_index(drop=True)

# Assign a rank
obs["rank"] = np.arange(1, len(obs) + 1)

# Assign a return period based on the rank (for 18 years we get RPs 1/18, 2/18 = 1/9, 3/18 = 1/6, ...)
obs["rp"] = N_YEARS_OBSERVATIONS / obs["rank"]

# ...except for the most severe event where we have better return period information from the literature
obs.iloc[0, obs.columns.get_loc("rp")] = RETURN_PERIOD_2017

# Index by return period
obs = obs[[
        "rp",
        "homes_affected_present",
        "homes_destroyed_present",
        "affected_or_destroyed_present",
        "recovery_cost_usd",
        "victims_present",
        "affected_present",
        "victims_or_affected_present"
    ]].set_index("rp")

In [26]:
# Write outputs to file

obs_residential = obs[["homes_affected_present", "homes_destroyed_present", "affected_or_destroyed_present", "recovery_cost_usd"]]
out_filepath_residential = Path(OBSERVATIONS_DIR) / "observations_by_rp_residential.csv"
obs_residential.to_csv(out_filepath_residential)

print("Output: residential impacts by return period")
display(obs_residential.style.format({
    "homes_affected_present": "{:,.1f}",
    "homes_destroyed_present": "{:,.1f}",
    "affected_or_destroyed_present": "{:,.1f}",
    "recovery_cost_usd": "${:,.0f}"
}))

obs_people = obs[["victims_present", "affected_present", "victims_or_affected_present"]]
out_filepath_people = Path(OBSERVATIONS_DIR) / "observations_by_rp_people.csv"
obs_people.to_csv(out_filepath_people)

print("Output: people impacts by return period")
display(obs_people.style.format({
    "victims_present": "{:,.1f}",
    "affected_present": "{:,.1f}",
    "victims_or_affected_present": "{:,.1f}"
}))


Output: residential impacts by return period


,homes_affected_present,homes_destroyed_present,affected_or_destroyed_present,recovery_cost_usd
rp,,,,
92.000000,"4,897.4",149.7,"5,047.1","$21,790,936"
9.000000,82.1,0.0,82.1,"$349,611"
6.000000,15.2,1.3,16.4,"$72,494"


Output: people impacts by return period


,victims_present,affected_present,victims_or_affected_present
rp,,,
92.000000,796.3,"24,487.1","25,283.5"
9.000000,0.0,413.9,413.9
6.000000,13.9,73.4,87.3
